# Notebook v3 beta class — train 200 ภาพต่อคลาส, epochs = 10

train **400 ภาพ (No Finding 200 / Infiltration-only 200)**: เก็บ 320 ภาพเดิมทั้งหมด
และเพิ่มคลาสละ 40 ภาพจากฝั่ง train ตาม patient partition เดิม ด้วย seed 42
validation 80 (40/40), test 100 (50/50), bbox XAI 123 ภาพ / 115 คนไข้ และ background 32 ภาพเดิม
อ่าน manifests ที่เตรียมไว้และตรวจ hashes ไม่สุ่ม split หรือเลือกภาพใหม่ตอนรัน

เปรียบเทียบ **Baseline / DWT L1–L4 + A4 ไม่ใช้ CLAHE / DFT ต่ำ–กลาง–สูง**
ทั้ง 3 วิธี freeze ResNet50 ImageNet V2 และ BatchNorm ทุกชั้น
ฝึกเฉพาะ `Dropout(0.3) → Linear(2048,1)` ด้วย AdamW weight_decay=1e-4
lr=1e-4, batch 16, seed 42, **epochs=10, patience=10** และ threshold 0.5

พารามิเตอร์ฝึกลดจากประมาณ 14.97 ล้านใน v2 เหลือ **2,049**
ใช้มาตรการบรรเทา overfit เดิมของ v3 beta ต้องฝึก baseline ใหม่ด้วย train 400 / epochs 10 เช่นกัน
ยังไม่มีผลวิจัยจาก notebook นี้ ไม่รับประกัน Accuracy เพิ่ม +10 จุดเปอร์เซ็นต์
ต้องอ่าน train_eval_loss/validation_loss ก่อนสรุปว่า overfit ลดลง
เทียบกับรุ่น e2/e7 ได้ในเชิงภาพรวม แต่จำนวนภาพและ epochs เปลี่ยนพร้อมกัน จึงแยกผลของแต่ละปัจจัยไม่ได้

## 1. Setup
Colab: อัปโหลด `Notebook_3_beta_class_colab.zip` ไป MyDrive เปิด notebook นี้ และเลือก GPU
ZIP แนบข้อมูลครบทั้ง 400 ภาพฝึกและชุดประเมินเดิม พร้อม manifests และโค้ด ไม่ต้องโหลด NIH ใหม่
Local: เลือก kernel `D:/kaggle_cache/datasets/nih-chest-xrays/.gpu/Scripts/python.exe`
เปิดจากโฟลเดอร์ `Notebook_3 beta class` ค่าตั้งทั้งหมดอยู่ใน `shared/config.json`
ผลเขียนลง `runs/class200_e10` แยกจาก e2/e7 ใช้ checkpoint ของ protocol นี้เพื่อ resume
`reference_run` เก็บ manifests เดิม 320 ภาพ; `source_run` เก็บ manifests 400 ภาพที่รันจริง

In [ ]:
from pathlib import Path
import sys, json, zipfile, shutil, subprocess, importlib, os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DATA_ROOT = None
OUTPUT_ROOT = None  # defaults to runs/class200_e10, separate from v2
SOURCE_RUN = None   # fixed expanded 400/80/100 manifests; never re-split at runtime
PACKAGE_ZIP = Path("/content/drive/MyDrive/Notebook_3_beta_class_colab.zip")

if IN_COLAB:
    drive.mount("/content/drive")
    NOTEBOOK_3_DIR = Path("/content/drive/MyDrive/Notebook_3_beta_class")
    if not PACKAGE_ZIP.is_file():
        raise FileNotFoundError("Upload Notebook_3_beta_class_colab.zip to MyDrive first")
    with zipfile.ZipFile(PACKAGE_ZIP) as archive:
        for member in archive.infolist():
            if member.is_dir() or member.filename == "Notebook_3_class/nih_cxr_subset.zip":
                continue
            relative = Path(member.filename)
            if relative.is_absolute() or ".." in relative.parts or relative.parts[0] != "Notebook_3_class":
                raise ValueError("Invalid package path")
            target = NOTEBOOK_3_DIR.joinpath(*relative.parts[1:]).resolve()
            if not target.is_relative_to(NOTEBOOK_3_DIR.resolve()):
                raise ValueError("Invalid extraction target")
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "PyWavelets",
                           "opencv-python-headless", "scikit-learn", "pandas", "matplotlib"])
    os.environ["CXR_PREPROCESS_CACHE_ROOT"] = "/content/nih_cxr_v3_cache"
else:
    candidates = [Path.cwd(), *Path.cwd().parents,
                  Path.cwd() / "Notebook_3 beta class", Path.cwd() / "Notebook_3_class"]
    NOTEBOOK_3_DIR = next((p for p in candidates if (p / "shared/beta.py").is_file()), None)
    if NOTEBOOK_3_DIR is None:
        raise FileNotFoundError("Run from Notebook_3 beta class or set NOTEBOOK_3_DIR")
    old_torch_cache = NOTEBOOK_3_DIR.parent / "Notebook_2 release/.runtime/torch"
    if old_torch_cache.is_dir():
        os.environ.setdefault("TORCH_HOME", str(old_torch_cache))

sys.path.insert(0, str(NOTEBOOK_3_DIR))
for name in tuple(sys.modules):
    if name == "shared" or name.startswith("shared."):
        del sys.modules[name]
importlib.invalidate_caches()

import numpy as np
import pandas as pd
import torch
from IPython.display import display, Image
from shared.experiment import Experiment, versions
from shared.frequency import open_frequency_experiment, FrequencyPreprocessor, frequency_reports
from shared.beta import train_beta, evaluate_beta, beta_model_factory, plot_training_history
if IN_COLAB and DATA_ROOT is None:
    from shared.data_bundle import colab_data_from_package
    DATA_ROOT = colab_data_from_package(PACKAGE_ZIP, NOTEBOOK_3_DIR,
                                       cache_root="/content/nih_cxr_v3_data")
if DATA_ROOT is None:
    DATA_ROOT = Experiment.open(NOTEBOOK_3_DIR).data_root
if SOURCE_RUN is None:
    SOURCE_RUN = NOTEBOOK_3_DIR / "source_run"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("Select a CUDA/GPU runtime before training or XAI")
exp, manifests = open_frequency_experiment(NOTEBOOK_3_DIR, DATA_ROOT, SOURCE_RUN, OUTPUT_ROOT)
print("Device:", torch.cuda.get_device_name(0), "| Data:", exp.data_root, "| Output:", exp.output_root)
print(versions())


## 2. ตรวจข้อมูลและยืนยันว่า freeze จริง
แสดงจำนวนคลาสและคนไข้จากไฟล์ที่รันจริง ไม่ใช้ class weighting เพราะชุด train สมดุล
ตรวจ metadata/official split/bbox/patient leakage และ hashes ก่อนเปิด experiment
model probe ไม่ใช้ pretrained download และไม่ฝึก

In [ ]:
display(pd.DataFrame([{"split": name, "images": len(frame),
                          "patients": frame.patient_id.nunique(),
                          "normal": int((frame.label == 0).sum()) if "label" in frame else None,
                          "infiltration": int((frame.label == 1).sum()) if "label" in frame else None}
                         for name, frame in manifests.items()]))
print(json.dumps(exp.config["cnn"], indent=2))
print("Inherited manifest hashes:", exp.config["frequency_fusion"]["source_manifest_hashes"])
selection = json.loads((SOURCE_RUN / "shared/artifacts/dataset_selection.json").read_text(encoding="utf-8"))
print("Training data expansion:", selection["retained_train_images"], "retained +",
      selection["added_train_images"], "added")
if exp.config["run_mode"] == "full":
    assert len(manifests["train"]) == 400
    assert manifests["train"].label.value_counts().to_dict() == {0: 200, 1: 200}
probe = beta_model_factory(exp)(pretrained=False, weights_name=exp.config["cnn"]["weights"])
probe.train()
trainable = {name: p.numel() for name, p in probe.named_parameters() if p.requires_grad}
assert sum(trainable.values()) == 2049
assert all(name.startswith("head.") for name in trainable)
assert all(not m.training for m in probe.backbone.modules())
assert not any(p.requires_grad for p in probe.backbone.parameters())
assert exp.config["cnn"]["epochs"] == 10
assert exp.config["cnn"]["patience"] >= 10
print("Trainable parameters:", trainable, "| Total:", sum(trainable.values()))
del probe

## 3. Feature fusion และ regularization

- Baseline: raw 1 view ใช้ preprocessing เดิม
- DWT: inverse DWT ของ A4 + D1/D2/D3/D4 (db1, periodization) เป็น 5 views
  ไม่ใช้ CLAHE และไม่ตัด coefficient; Dj รวมรายละเอียด 3 ทิศทาง
- DFT: radial low <0.125, mid 0.125–<0.25, high ≥0.25 cycles/pixel แล้ว inverse FFT เป็น 3 views
  masks ไม่ซ้อนกันและครอบคลุมทุก coefficient

`views → ResNet50 (frozen/eval) → mean(feature 2048) → Dropout(0.3) → Linear → logit`

Dropout ใช้ **หลังเฉลี่ย feature** เป็นหนึ่ง mask ต่อภาพต้นฉบับทุกวิธี
ไม่ใช้ dropout แยกตาม view ซึ่งจะเปลี่ยนความแรง regularization ตามจำนวน views
ตอน validation/test/XAI ปิด dropout และใช้ BatchNorm statistics ของ pretrained backbone
optimizer เห็นเฉพาะ head; backbone ไม่มี gradient ของ weights แต่ยังหา gradient ของ input สำหรับ XAI ได้

ทำงานที่ 256×256/resize 224×224/RGB/ImageNet mean/std เดิม
low/A4 ใช้ค่าเดิม; signed detail ใช้ 0.5+0.5×component ไม่มี clipping หรือ min-max ต่อภาพ
ไม่มี augmentation หรือ class weighting เพิ่มใน beta นี้ เพื่อคง preprocessing เดิม
batch 16 ภาพเป็น 80 DWT views/48 DFT views; จำนวนภาพและ hyperparameters เท่ากัน แต่ compute ต่างกัน

In [ ]:
import matplotlib.pyplot as plt
row = manifests["train"].iloc[0].to_dict()
names = {"baseline": ["raw"], "dwt_L1_L4_fusion": ["A4", "D1", "D2", "D3", "D4"],
         "dft_low_mid_high_fusion": ["low", "mid", "high"]}
for condition_id in exp.condition_ids:
    prep = FrequencyPreprocessor(exp, condition_id)
    bands = prep.components(row)
    fig, axes = plt.subplots(1, len(bands), figsize=(4*len(bands), 4), squeeze=False)
    for ax, band, name in zip(axes[0], bands, names[condition_id]):
        ax.imshow(band, cmap="gray" if name in ("raw", "A4", "low") else "seismic")
        ax.set_title(name); ax.axis("off")
    fig.suptitle(condition_id + " — display autoscaling only")
    display(fig); plt.close(fig)
    print(condition_id, "input shape:", tuple(prep.tensor(row).shape))
del prep

## 4. CNN: ฝึกทั้ง 3 วิธีครบ 10 epochs
เซลล์นี้ฝึกจริงจาก ImageNet initialization แบบเดียวกันทุกวิธี และบันทึก best.pt/last.pt
ตั้ง patience=10 จึงไม่หยุดก่อนครบ 10 รอบ แม้ validation loss แย่ลงตั้งแต่ epoch 2
เลือก best epoch จากรอบ 1–10 ด้วย validation loss; ไม่ได้บังคับใช้ checkpoint รอบ 10 ประเมินผล
ไม่ดู test เพื่อเลือก checkpoint/threshold และไม่ใช้ผล baseline 54% ของ v2 แทน baseline v3

RTX 4060 ประเมิน training + classification ประมาณ 2–10 นาที ไม่รวม download ครั้งแรก
ยังไม่ได้วัดการฝึกเต็มรุ่นนี้; XAI โดยเฉพาะ SHAP ใช้เวลามากกว่า
รันซ้ำ resume ได้ หาก config/code/versions เปลี่ยนให้ใช้ OUTPUT_ROOT ใหม่

In [ ]:
for condition_id in exp.condition_ids:
    metrics = train_beta(exp, manifests, condition_id, DEVICE)
    assert metrics["epochs_run"] == 10 and metrics["trainable_parameters"] == 2049
    print(condition_id, {k: metrics[k] for k in ("accuracy", "precision", "recall", "f1", "roc_auc", "best_epoch")})
    cnn_table, xai_table, paired_deltas, status = frequency_reports(exp)
    display(status)
display(cnn_table)

## 5. ตรวจ train/validation gap
`train_loss` ระหว่าง update เปิด dropout และเปลี่ยนน้ำหนักไปแต่ละ batch
จึงบันทึก **train_eval_loss** หลังจบ epoch ด้วย model.eval() เช่นเดียวกับ validation
ใช้ train_eval_loss เทียบ validation_loss เพื่ออ่าน gap; ไม่ใช้ train Accuracy ว่าเป็น test Accuracy
ดูแนวโน้มตลอด 10 รอบ: train_eval_loss ลดแต่ validation_loss เพิ่มต่อเนื่องเป็นสัญญาณ overfit
หากทั้งสอง loss สูงอาจเป็น underfit; freeze/regularization ไม่รับประกันว่าจะขจัดปัญหาได้ทั้งหมด

In [ ]:
curve_path = plot_training_history(exp)
display(Image(filename=str(curve_path)))
for condition_id in exp.condition_ids:
    print(condition_id)
    display(pd.read_csv(exp.results("CNN", condition_id) / "history.csv"))

## 6. Grad-CAM: bbox 123 ภาพเดิม
อธิบาย Infiltration logit ผ่าน layer4 ของ frozen backbone ทุก view
เปิด gradient ที่ input และรวม signed contributions ก่อน ReLU; ไม่ใช้ no_grad ครอบ XAI
ใช้ checkpoint ที่เลือกจาก validation และอธิบายทุกภาพ รวมคำทำนายผิด
รายงาน IoU@0.3/0.5, Pointing Game และ energy inside; heatmap ว่างเป็น miss/IoU 0

In [ ]:
for condition_id in exp.condition_ids:
    records = evaluate_beta(exp, manifests, "Grad-CAM", condition_id, DEVICE)
    print(condition_id, "evaluated:", len(records))
    frequency_reports(exp)

## 7. SHAP Expected Gradients
background train IDs เดิม 32 ภาพ; nsamples=200, batch_size=8, local_smoothing=0
บันทึก signed attribution ของทุก view/RGB ก่อนรวม ใช้ส่วนบวกประเมิน bbox
อธิบาย processed bands ไม่ใช่ raw-pixel gradients ผ่าน NumPy DWT/FFT
อาจมี ringing/สัญญาณกระจายจาก frequency filters จึงไม่ตีความเป็น lesion segmentation

Training + XAI ครบชุดบน RTX 4060 ประเมินเผื่อ 45–90 นาทีจากการวัด pipeline v2 ก่อนหน้า
ยังไม่ใช่ benchmark เต็มของ v3; เวลา Colab ขึ้นกับ GPU เซลล์นี้ resume รายภาพได้

In [ ]:
for condition_id in exp.condition_ids:
    records = evaluate_beta(exp, manifests, "SHAP", condition_id, DEVICE)
    print(condition_id, "evaluated:", len(records))
    frequency_reports(exp)

## 8. Classification / XAI และเกณฑ์ +10 จุดเปอร์เซ็นต์
ค่าความต่างต้องเทียบ baseline train 400 / epochs 10 ใน protocol นี้: target = baseline_v3_class + 0.10
0.10 บนสเกล 0–1 คือ 10 จุดเปอร์เซ็นต์ ไม่ใช่ relative improvement 10%
XAI ใช้ patient bootstrap 2,000 ครั้ง รวม overall/pure/mixed และ paired deltas/95% CI
classification delta เป็น descriptive; ไม่อ้าง significance จากคะแนนเพิ่มเพียงอย่างเดียว
test 100 ภาพเป็น pilot เดิม ไม่ใช่ official binary test เต็ม 12,081 ภาพ

In [ ]:
cnn_table, xai_table, paired_deltas, status = frequency_reports(exp)
display(status); display(cnn_table); display(xai_table); display(paired_deltas)
base_acc = float(cnn_table.set_index("condition_id").loc["baseline", "accuracy"])
acceptance = cnn_table[["condition_id", "accuracy", "delta_accuracy_vs_baseline"]].copy()
acceptance["accuracy_target"] = base_acc + 0.10
acceptance["meets_plus_10_percentage_points"] = acceptance.delta_accuracy_vs_baseline >= (0.10 - 1e-12)
acceptance.to_csv(exp.output_root / "comparison/accuracy_target.csv", index=False)
display(acceptance)
for condition_id in exp.condition_ids:
    display(Image(filename=str(exp.results("CNN", condition_id) / "confusion_matrix.png")))
example = manifests["xai"].iloc[0]["image"].rsplit(".", 1)[0]
for criterion in ("Grad-CAM", "SHAP"):
    for condition_id in exp.condition_ids:
        print(criterion, condition_id)
        display(Image(filename=str(exp.results(criterion, condition_id) / "overlays" / (example + ".png"))))
print("Measured outputs:", exp.output_root)

## 9. Export ผลจริง
รวม checkpoints/history/predictions/manifests/XAI/ตารางและกราฟ ไม่รวม image cache
ผลหลักเก็บใน Drive แม้ Colab disconnect ผล smoke test ไม่อยู่ใน ZIP นี้

In [ ]:
export_path = exp.output_root.parent / "Notebook_3_beta_class_results.zip"
with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in exp.output_root.rglob("*"):
        if path.is_file() and "cache" not in path.relative_to(exp.output_root).parts:
            archive.write(path, path.relative_to(exp.output_root).as_posix())
print(export_path)
if IN_COLAB:
    from google.colab import files
    files.download(str(export_path))